In [ ]:
import torch
import random
import math


/home/ravi/envs/.venv/lib/python3.14/site-packages/torch/_subclasses/functional_tensor.py:283: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [ ]:
class microg():
    def __init__(self, value,t = None,opp=None , ch1 = None , ch2 = None):
        value = float(value)
        self.value = value
        if t is not None:
            self.t = t
        else:
            self.t = torch.tensor(value,requires_grad = True)
        self.opperation = opp
        self.ch1 = ch1
        self.ch2 = ch2 
        self.grad = 0

    @property
    def torch_grad(self):
        return self.t.grad


    def __repr__(self):
        if self.opperation:
            return(f"value: {self.value} \nopperation: {self.opperation} \nchild1: {self.ch1.value} \nchild2: {self.ch2.value} \ngrad: {self.grad} \ntorch_grad: {self.torch_grad} \ntorch_val: {self.t}")
        else:
            return(f"value: {self.value} \ngrad: {self.grad} \ntorch_grad: {self.torch_grad} \ntorch_val: {self.t}")

    def __add__ (self,value):
        if type(value) != microg:
            value = microg(value)
        out = microg(self.value + value.value,self.t+value.t ,'+',self,value)
        return out


    def __mul__ (self,value):
        if type(value) != microg:
            value = microg(value)
        out = microg(self.value * value.value,self.t*value.t ,'*',self,value)
        return out


    def __sub__ (self,value):
        value = value * -1
        out = self + value
        return out

    def tanh(self):
        t_val = math.tanh(self.value)
        out = microg(t_val, self.t.tanh(), 'tanh', self, self)
        return out





    def backward(self, carry=None):
        if carry is None:
            self.grad = 1.0
            self.t.backward()
            carry = 1
        
        if self.opperation:


            if self.opperation == '+':
                self.ch1.backward(carry)
                self.ch2.backward(carry)
                self.ch1.grad += carry
                self.ch2.grad += carry

            if self.opperation == '*':
                self.ch1.backward(carry*self.ch2.value)
                self.ch2.backward(carry*self.ch1.value)
                self.ch1.grad += carry*self.ch2.value
                self.ch2.grad += carry*self.ch1.value


            if self.opperation == 'tanh':
                local_grad = 1 - (self.value ** 2)
                self.ch1.grad += carry * local_grad
                self.ch1.backward(carry * local_grad)




    def zero_grad(self):
        self.grad = 0
        self.t.grad = None
        if self.opperation:
            self.ch1.zero_grad()
            self.ch2.zero_grad()



In [3]:
a = microg(10.0)
b = microg(5.0)
c = a+b
d = c*a
d.backward()
a

value: 10.0 
grad: 25.0 
torch_grad: 25.0 
torch_val: 10.0

In [4]:
a = microg(2.0)
b = microg(0.0)
c = a * b       # c = 0
d = c + a       # d = 2
d.backward()
print(f"a.grad = {a.grad}, torch says = {a.torch_grad}")
print(f"b.grad = {b.grad}, torch says = {b.torch_grad}")


a.grad = 1.0, torch says = 1.0
b.grad = 2.0, torch says = 2.0


In [5]:
a = torch.tensor(2.0,requires_grad=True)
b = torch.tensor(0.0,requires_grad=True)
c = a * b       # c = 0
d = c + a       # d = 2
d.backward()
b.grad

tensor(2.)

# MLP

In [500]:
class neuron():
    def __init__(self, no_of_features:int):
        rand_numb = []
        for i in range(no_of_features):
            # random.seed(4)
            # rand_numb.append(microg(random.randint(1,5)))
            rand_numb.append(microg(random.random()))
        self.weights = rand_numb
        # random.seed(4)

        # self.bias = microg(random.randint(1,5))
        self.bias = microg(random.random())


    def execute(self,feature:list):
        out = microg(0)
        for i,j in zip(feature,self.weights):
            if type(i) is not microg:
                i = microg(i)
            out += i*j
        return out+self.bias

In [501]:
class layer():
    def __init__(self,no_of_nurons,no_of_features):
        nu = []
        weights =[]
        for i in range(no_of_nurons):
            x = neuron(no_of_features)
            weights.extend(x.weights)
            weights.append(x.bias)
            nu.append(x)
        self.weights=weights
        self.nurons= nu
        
    def execute(self,feature:list):
        out = []
        for i in self.nurons:
            out.append(i.execute(feature))

        return out

In [502]:
class mlp():
    """no of layers,no of nurons in each layer,features"""
    def __init__(self,no_of_layers,no_of_nurons,output_layer_nurons,no_of_features):
        weights = []
        layers = []

        x = layer(no_of_nurons,no_of_features)
        if no_of_nurons != 0:
            weights.extend(x.weights)
            layers.append(x)

        for i in range(no_of_layers-1):
            x = layer(no_of_nurons,no_of_nurons)
            weights.extend(x.weights)
            layers.append(x)

        if no_of_nurons == 0:
            x = layer(output_layer_nurons,no_of_features)
        else:
            x = layer(output_layer_nurons,no_of_nurons)

        weights.extend(x.weights)
        layers.append(x)
        self.weights = weights
        self.layers = layers

    def input(self,a:list):
        for i in self.layers:
            a = i.execute(a)
        return a
            
        

In [503]:
x = mlp(0,0,1,1)
print(x.input([22]))
print([i.value for i in x.weights])

[value: 12.58943964338203 
opperation: + 
child1: 11.82289384019525 
child2: 0.7665458031867807 
grad: 0 
torch_grad: None 
torch_val: 12.589439392089844]
[0.5374042654634205, 0.7665458031867807]


/tmp/ipykernel_51055/403401025.py:16: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:492.)
  return self.t.grad


In [509]:
dataset = []
for i in range(1,100):
    r = [[i],50*i]
    dataset.append(r)
print(dataset[:3])


[[[1], 50], [[2], 100], [[3], 150]]


In [512]:
# (self,no_of_layers,no_of_nurons,output_layer_nurons,no_of_features):
x = mlp(3,5,1,1)


In [550]:
t1 = 53
temp = x.input([t1])

print(f'{temp[0].value} ---> {t1*50}')

2657.713346891279 ---> 2650


In [551]:
# x = mlp(3,5,1,1)


for _ in range(100):
    loss.zero_grad()

    loss = microg(0)
    for i in dataset:
        # print(i)
        temp = x.input(i[0])
        # print(temp)
        t2 = temp[0] - i[1]
        t2 = t2*t2
        loss += t2

    # print(f"===> {[i.value for i in x.weights]}")
    print(loss.value)
    print("--------------------")
    loss.backward()

    for i in x.weights:
        i.value = i.value - (i.grad*0.000000001)



    

36539.943181608585
--------------------
36536.95995098137
--------------------
36533.97706415083
--------------------
36530.994521059794
--------------------
36528.01232165028
--------------------
36525.03046586676
--------------------
36522.048953651814
--------------------
36519.0677849492
--------------------
36516.086959701344
--------------------
36513.10647785149
--------------------
36510.12633934387
--------------------
36507.14654412077
--------------------
36504.16709212527
--------------------
36501.18798330137
--------------------
36498.20921759243
--------------------
36495.23079494116
--------------------
36492.252715291135
--------------------
36489.274978584785
--------------------
36486.29758476712
--------------------
36483.320533780454
--------------------
36480.34382556809
--------------------
36477.36746007361
--------------------
36474.39143724078
--------------------
36471.415757012284
--------------------
36468.440419331906
--------------------
36465.46542414315

In [595]:
loss = microg(0)
for i in dataset:
    # print(i)
    temp = x.input(i[0])
    # print(temp)
    t2 = temp[0] - i[1]
    t2 = t2*t2
    loss += t2

loss.backward()

In [596]:
x.weights

[value: 3.0 
 grad: -1303500.0 
 torch_grad: -1303500.0 
 torch_val: 3.0,
 value: 1.0 
 grad: -19602.0 
 torch_grad: -19602.0 
 torch_val: 1.0]

In [591]:
temp

/tmp/ipykernel_2143/403401025.py:16: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more information. (Triggered internally at /pytorch/build/aten/src/ATen/core/TensorBody.h:492.)
  return self.t.grad


[value: 6.0 
 opperation: + 
 child1: 5.0 
 child2: 1.0 
 grad: 2.0 
 torch_grad: None 
 torch_val: 6.0]